# **IMPORT LIBRARY**

In [1]:
import pandas as pd
import numpy as np
import os
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [2]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

RAW_DIR = Path("../../../data/raw")
OUTPUT_DIR = Path("../../../data/processed/02_lstm_ae_personalized_model")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WINDOW_SIZE = 14
MIN_DAYS_REQUIRED = 14

print(f"RAW_DIR   : {RAW_DIR.resolve()}")
print(f"OUTPUT_DIR: {OUTPUT_DIR.resolve()}")
print(f"WINDOW_SIZE       : {WINDOW_SIZE}")
print(f"MIN_DAYS_REQUIRED : {MIN_DAYS_REQUIRED}")

RAW_DIR   : /Users/faqihfirmanpratama/Documents/DBS Codingcamp/capstone/AI-ENGINEER/data/raw
OUTPUT_DIR: /Users/faqihfirmanpratama/Documents/DBS Codingcamp/capstone/AI-ENGINEER/data/processed/02_lstm_ae_personalized_model
WINDOW_SIZE       : 14
MIN_DAYS_REQUIRED : 14


# **LOAD DATA**

In [ ]:
df_transaksi = pd.read_csv( RAW_DIR / "data_transaksi.csv")
df_nasabah   = pd.read_csv(RAW_DIR / "data_nasabah.csv")
df_features  = pd.read_csv(RAW_DIR / "features_nasabah.csv")

In [17]:
display(df_transaksi.head())
print("=" * 60)
print("data_transaksi.csv")
print(f"shape  : {df_transaksi.shape}\n")
print(f"dtypes :")
print(df_transaksi.info())

,id_transaksi,id_user,timestamp,tipe_mutasi,deskripsi_mutasi,catatan_mutasi,mcc,nominal,sisa_saldo,gt_kategori_besar,...,hari_minggu,jam,menit,hari_bulan,hour_sin,hour_cos,month_sin,month_cos,kategori_detail,kategori_besar
0,TRX-10139,USR-001,2026-01-01 07:00:00,Kredit,TRF MASUK - PENDAPATAN TETAP,-,0,3130000.00,3130000.00,Pemasukan,...,3,7,0,1,9.659258e-01,-0.258819,0.0,1.0,Pendapatan Bulanan,Income
1,TRX-10140,USR-001,2026-01-01 09:00:00,Debit,NETFLIX,Auto-Debit Monthly,4899,186000.00,2944000.00,Wants,...,3,9,0,1,7.071068e-01,-0.707107,0.0,1.0,Hiburan & Langganan,Wants
2,TRX-10141,USR-001,2026-01-01 09:00:00,Debit,BPJS KESEHATAN,Auto-Debit Monthly,8099,150000.00,2794000.00,Needs,...,3,9,0,1,7.071068e-01,-0.707107,0.0,1.0,Kesehatan & Perawatan Diri,Needs
3,TRX-11790,USR-001,2026-01-01 10:00:00,Debit,STEAM/VALORANT,-,7999,112017.17,2681982.83,Wants,...,3,10,0,1,5.000000e-01,-0.866025,0.0,1.0,Hiburan & Langganan,Wants
4,TRX-11792,USR-001,2026-01-01 12:00:00,Debit,UNIQLO INDONESIA,-,5651,595635.11,1547667.73,Wants,...,3,12,0,1,1.224647e-16,-1.000000,0.0,1.0,Belanja Online & Fashion,Wants


data_transaksi.csv
shape  : (53146, 23)

dtypes :
<class 'pandas.DataFrame'>
RangeIndex: 53146 entries, 0 to 53145
Data columns (total 23 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_transaksi        53146 non-null  str    
 1   id_user             53146 non-null  str    
 2   timestamp           53146 non-null  str    
 3   tipe_mutasi         53146 non-null  str    
 4   deskripsi_mutasi    53146 non-null  str    
 5   catatan_mutasi      53146 non-null  str    
 6   mcc                 53146 non-null  int64  
 7   nominal             53146 non-null  float64
 8   sisa_saldo          53146 non-null  float64
 9   gt_kategori_besar   53146 non-null  str    
 10  gt_kategori_detail  53146 non-null  str    
 11  label_anomali       53146 non-null  int64  
 12  bulan               53146 non-null  int64  
 13  hari_minggu         53146 non-null  int64  
 14  jam                 53146 non-null  int64  
 15  menit         

In [18]:
display(df_nasabah.head())

print("=" * 60)
print("data_nasabah.csv")
print(f"shape  : {df_nasabah.shape}\n")
print(f"dtypes :")
print(df_nasabah.info())


,id_user,nama_nasabah,tanggal_lahir,nama_ibu_kandung,segmen_demografi,gaji_bulanan,persona_dasar,is_dynamic,persona_bulan_1,persona_bulan_2,persona_bulan_3
0,USR-362,Sutan Perkasa Hakim,1992-10-30,Gina Permata,First Jobber,4750000,Unconflicted,0,Unconflicted,Unconflicted,Unconflicted
1,USR-074,"Bakiadi Suryono, M.TI.",2002-04-04,dr. Cornelia Prakasa,Mahasiswa,3210000,Unconflicted,0,Unconflicted,Unconflicted,Unconflicted
2,USR-375,Karen Mulyani,2006-03-14,Cinthia Halimah,First Jobber,5580000,Tightwad,0,Tightwad,Tightwad,Tightwad
3,USR-156,Sari Laksita,2006-01-04,Lasmanto Hidayat,Mahasiswa,2970000,Spendthrift,0,Spendthrift,Spendthrift,Spendthrift
4,USR-105,"drg. Jaka Fujiati, M.Farm",1995-02-20,"R.A. Yance Irawan, S.T.",Mahasiswa,1600000,Tightwad,1,Tightwad,Tightwad,Tightwad


data_nasabah.csv
shape  : (500, 11)

dtypes :
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   id_user           500 non-null    str  
 1   nama_nasabah      500 non-null    str  
 2   tanggal_lahir     500 non-null    str  
 3   nama_ibu_kandung  500 non-null    str  
 4   segmen_demografi  500 non-null    str  
 5   gaji_bulanan      500 non-null    int64
 6   persona_dasar     500 non-null    str  
 7   is_dynamic        500 non-null    int64
 8   persona_bulan_1   500 non-null    str  
 9   persona_bulan_2   500 non-null    str  
 10  persona_bulan_3   500 non-null    str  
dtypes: int64(2), str(9)
memory usage: 43.1 KB
None


In [19]:
display(df_features.head())

print("=" * 60)
print("features_nasabah.csv")
print(f"shape  : {df_features.shape}\n")
print(f"dtypes :")
print(df_features.info())

,id_user,bulan,wants_ratio,fixed_costs_ratio,savings_rate,wants_frequency,small_leaks_ratio,night_owl_spending,weekend_surge,early_month_depletion,balance_volatility,survival_mode_days
0,USR-001,1,0.900940,0.099060,0.001871,0.722222,0.333333,0.111111,0.009067,0.995223,0.200906,4
1,USR-001,2,0.170089,0.522909,0.001690,0.333333,0.166667,0.000000,7.774289,0.989353,0.281796,4
2,USR-001,3,0.059316,0.940684,-0.001833,0.142857,0.285714,0.000000,1.000000,0.995125,0.003570,3
3,USR-002,1,0.313902,0.463219,0.012319,0.380952,0.142857,0.047619,0.141356,0.970751,0.227404,4
4,USR-002,2,0.699313,0.300687,-0.009408,0.708333,0.291667,0.041667,2.084027,1.009408,0.327462,1


features_nasabah.csv
shape  : (1500, 12)

dtypes :
<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id_user                1500 non-null   str    
 1   bulan                  1500 non-null   int64  
 2   wants_ratio            1500 non-null   float64
 3   fixed_costs_ratio      1500 non-null   float64
 4   savings_rate           1500 non-null   float64
 5   wants_frequency        1500 non-null   float64
 6   small_leaks_ratio      1500 non-null   float64
 7   night_owl_spending     1500 non-null   float64
 8   weekend_surge          1500 non-null   float64
 9   early_month_depletion  1500 non-null   float64
 10  balance_volatility     1500 non-null   float64
 11  survival_mode_days     1500 non-null   int64  
dtypes: float64(9), int64(2), str(1)
memory usage: 140.8 KB
None


In [ ]:
print("Total unique users           :", df_transaksi['id_user'].nunique())
print("Total unique kategori_detail :", df_transaksi['kategori_detail'].nunique())
print("Rentang tanggal transaksi    :", pd.to_datetime(df_transaksi['timestamp']).dt.date.min(), "s/d",pd.to_datetime(df_transaksi['timestamp']).dt.date.max())   

Total unique users           : 500
Total unique kategori_detail : 12
Rentang tanggal transaksi    : 2026-01-01 s/d 2026-03-31


# **FILTER & CLEAN TRANSACTION**

In [22]:
n_before = len(df_transaksi)
df_debit = df_transaksi[df_transaksi['tipe_mutasi'] == 'Debit'].copy()
n_after  = len(df_debit)

print(f"Baris sebelum filter    : {n_before:,}")
print(f"Baris setelah filter    : {n_after:,}")
print(f"Baris dihapus           : {n_before - n_after:,}")

Baris sebelum filter    : 53,146
Baris setelah filter    : 50,730
Baris dihapus           : 2,416


In [23]:
df_debit['timestamp'] = pd.to_datetime(df_debit['timestamp'])
df_debit['tanggal']   = df_debit['timestamp'].dt.normalize()

In [24]:
df_clean = df_debit[['id_user', 'tanggal', 'kategori_detail', 'nominal']].copy()
print(f"Kolom: {df_clean.columns.tolist()}")
print(f"Shape: {df_clean.shape}")

Kolom: ['id_user', 'tanggal', 'kategori_detail', 'nominal']
Shape: (50730, 4)


In [ ]:
# Sanity check
print("Null counts:")
print(df_clean.isnull().sum())
print()
print(f"Min tanggal : {df_clean['tanggal'].min().date()}")
print(f"Max tanggal : {df_clean['tanggal'].max().date()}")
print()
print("Unique kategori_detail (sorted):")
for k in sorted(df_clean['kategori_detail'].unique()):
    print(f"* {k}")

Null counts:
id_user            0
tanggal            0
kategori_detail    0
nominal            0
dtype: int64

Min tanggal : 2026-01-01
Max tanggal : 2026-03-31

Unique kategori_detail (sorted):
* Belanja Online & Fashion
* F&B dan Nongkrong
* Groceries & Kebutuhan Pokok
* Hiburan & Langganan
* Investasi & Finansial
* Kesehatan & Perawatan Diri
* NLP Classified Transfer
* Produktivitas & Digital
* Tagihan & Utilitas
* Transportasi


# **SPANDING MATRIX (PIVOT)**

In [27]:
#  Pivot: setiap baris = (id_user, tanggal), kolom = kategori spending
df_pivot = pd.pivot_table(
    df_clean,
    values='nominal',
    index=['id_user', 'tanggal'],
    columns='kategori_detail',
    aggfunc='sum',
    fill_value=0
).reset_index()

df_pivot.columns.name = None
print(f"Shape pivot table: {df_pivot.shape}")
display(df_pivot.head(3))

Shape pivot table: (14536, 12)


,id_user,tanggal,Belanja Online & Fashion,F&B dan Nongkrong,Groceries & Kebutuhan Pokok,Hiburan & Langganan,Investasi & Finansial,Kesehatan & Perawatan Diri,NLP Classified Transfer,Produktivitas & Digital,Tagihan & Utilitas,Transportasi
0,USR-001,2026-01-01,1134315.10,0.00,0.0,417144.84,0.0,150000.00,73294.82,0.00,0.0,0.0
1,USR-001,2026-01-02,1009947.77,16558.98,0.0,0.00,0.0,143320.53,41143.88,122258.31,0.0,0.0
2,USR-001,2026-01-03,0.00,0.00,7065.3,0.00,0.0,0.00,0.00,0.00,0.0,0.0


In [28]:
# Reindex per user untuk mengisi hari tanpa transaksi dengan 0 (Crucial agar LSTM menerima sequence)

category_cols = [c for c in df_pivot.columns if c not in ['id_user', 'tanggal']]

user_dfs = []
for user_id, grp in df_pivot.groupby('id_user'):
    grp = grp.set_index('tanggal').drop(columns='id_user')
    full_range = pd.date_range(grp.index.min(), grp.index.max(), freq='D')
    grp = grp.reindex(full_range, fill_value=0)
    grp['id_user'] = user_id
    user_dfs.append(grp)

df_daily = pd.concat(user_dfs)
df_daily = df_daily.reset_index().rename(columns={'index': 'tanggal'})

print(f"Shape setelah reindex: {df_daily.shape}")
display(df_daily.head(3))

Shape setelah reindex: (40523, 12)


,tanggal,Belanja Online & Fashion,F&B dan Nongkrong,Groceries & Kebutuhan Pokok,Hiburan & Langganan,Investasi & Finansial,Kesehatan & Perawatan Diri,NLP Classified Transfer,Produktivitas & Digital,Tagihan & Utilitas,Transportasi,id_user
0,2026-01-01,1134315.10,0.00,0.0,417144.84,0.0,150000.00,73294.82,0.00,0.0,0.0,USR-001
1,2026-01-02,1009947.77,16558.98,0.0,0.00,0.0,143320.53,41143.88,122258.31,0.0,0.0,USR-001
2,2026-01-03,0.00,0.00,7065.3,0.00,0.0,0.00,0.00,0.00,0.0,0.0,USR-001


In [30]:
days_per_user = df_daily.groupby('id_user').size()

print(f"Total baris daily matrix   : {len(df_daily):,}")
print(f"Rata-rata hari per user    : {days_per_user.mean():.1f}")

users_below_min = days_per_user[days_per_user < MIN_DAYS_REQUIRED]
print(f"\nUser dengan < {MIN_DAYS_REQUIRED} hari (tidak bisa membentuk window): {len(users_below_min)}")
if len(users_below_min) > 0:
    for uid, cnt in users_below_min.items():
        print(f"  {uid}: {cnt} hari")

print("\nSample USR-001 (10 baris pertama):")
display(df_daily[df_daily['id_user'] == 'USR-001'].head())

Total baris daily matrix   : 40,523
Rata-rata hari per user    : 81.0

User dengan < 14 hari (tidak bisa membentuk window): 0

Sample USR-001 (10 baris pertama):


,tanggal,Belanja Online & Fashion,F&B dan Nongkrong,Groceries & Kebutuhan Pokok,Hiburan & Langganan,Investasi & Finansial,Kesehatan & Perawatan Diri,NLP Classified Transfer,Produktivitas & Digital,Tagihan & Utilitas,Transportasi,id_user
0,2026-01-01,1134315.10,0.00,0.0,417144.84,0.0,150000.00,73294.82,0.00,0.0,0.0,USR-001
1,2026-01-02,1009947.77,16558.98,0.0,0.00,0.0,143320.53,41143.88,122258.31,0.0,0.0,USR-001
2,2026-01-03,0.00,0.00,7065.3,0.00,0.0,0.00,0.00,0.00,0.0,0.0,USR-001
3,2026-01-04,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.00,0.0,0.0,USR-001
4,2026-01-05,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.00,0.0,0.0,USR-001


# **ADD PROFILE FEATURES**

In [ ]:
# ordinal encoding
segmen_map  = {'Mahasiswa': 0, 'First Jobber': 1, 'Profesional': 2}
persona_map = {'Tightwad': 0, 'Unconflicted': 1, 'Spendthrift': 2}

df_profile = df_nasabah[['id_user', 'gaji_bulanan', 'segmen_demografi', 'persona_dasar']].copy()
df_profile['segmen_demografi_enc'] = df_profile['segmen_demografi'].map(segmen_map)
df_profile['persona_dasar_enc']    = df_profile['persona_dasar'].map(persona_map)
df_profile = df_profile[['id_user', 'gaji_bulanan', 'segmen_demografi_enc', 'persona_dasar_enc']]

print("Distribusi segmen_demografi_enc:")
print(df_profile['segmen_demografi_enc'].value_counts().sort_index())
print("\nDistribusi persona_dasar_enc:")
print(df_profile['persona_dasar_enc'].value_counts().sort_index())

Distribusi segmen_demografi_enc:
segmen_demografi_enc
0    200
1    200
2    100
Name: count, dtype: int64

Distribusi persona_dasar_enc:
persona_dasar_enc
0    159
1    167
2    174
Name: count, dtype: int64


In [34]:
# Left-join profile ke daily matrix
df = df_daily.merge(df_profile, on='id_user', how='left')

print(f"Shape setelah merge: {df.shape}")
display(df.head())

Shape setelah merge: (40523, 15)


,tanggal,Belanja Online & Fashion,F&B dan Nongkrong,Groceries & Kebutuhan Pokok,Hiburan & Langganan,Investasi & Finansial,Kesehatan & Perawatan Diri,NLP Classified Transfer,Produktivitas & Digital,Tagihan & Utilitas,Transportasi,id_user,gaji_bulanan,segmen_demografi_enc,persona_dasar_enc
0,2026-01-01,1134315.10,0.00,0.0,417144.84,0.0,150000.00,73294.82,0.00,0.0,0.0,USR-001,3130000,0,2
1,2026-01-02,1009947.77,16558.98,0.0,0.00,0.0,143320.53,41143.88,122258.31,0.0,0.0,USR-001,3130000,0,2
2,2026-01-03,0.00,0.00,7065.3,0.00,0.0,0.00,0.00,0.00,0.0,0.0,USR-001,3130000,0,2
3,2026-01-04,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.00,0.0,0.0,USR-001,3130000,0,2
4,2026-01-05,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.00,0.0,0.0,USR-001,3130000,0,2


In [35]:
# Verifikasi: tidak boleh ada NaN di kolom profile
profile_check = ['gaji_bulanan', 'segmen_demografi_enc', 'persona_dasar_enc']
null_counts   = df[profile_check].isnull().sum()
assert null_counts.sum() == 0, f"Ada NaN di kolom profile!\n{null_counts}"

print("Assertion passed: tidak ada NaN di kolom profile.")
print("\nStatistik gaji_bulanan:")
print(df['gaji_bulanan'].describe())

Assertion passed: tidak ada NaN di kolom profile.

Statistik gaji_bulanan:
count    4.052300e+04
mean     6.048819e+06
std      4.644546e+06
min      1.500000e+06
25%      2.870000e+06
50%      5.000000e+06
75%      5.770000e+06
max      1.998000e+07
Name: gaji_bulanan, dtype: float64


# **CYCLICAL TIME FEATURES**

In [36]:
df['day_of_week']  = df['tanggal'].dt.dayofweek   # 0=Senin, 6=Minggu
df['day_of_month'] = df['tanggal'].dt.day         # 1-31

df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
df['dom_sin'] = np.sin(2 * np.pi * df['day_of_month'] / 31)
df['dom_cos'] = np.cos(2 * np.pi * df['day_of_month'] / 31)

df.drop(columns=['day_of_week', 'day_of_month'], inplace=True)

In [37]:
sample_cols = ['tanggal', 'dow_sin', 'dow_cos', 'dom_sin', 'dom_cos']
print("Sample 7 baris USR-001 (verifikasi nilai siklus):")
print(df[df['id_user'] == 'USR-001'][sample_cols].head(7).to_string(index=False))

Sample 7 baris USR-001 (verifikasi nilai siklus):
   tanggal   dow_sin   dow_cos  dom_sin  dom_cos
2026-01-01  0.433884 -0.900969 0.201299 0.979530
2026-01-02 -0.433884 -0.900969 0.394356 0.918958
2026-01-03 -0.974928 -0.222521 0.571268 0.820763
2026-01-04 -0.781831  0.623490 0.724793 0.688967
2026-01-05  0.000000  1.000000 0.848644 0.528964
2026-01-06  0.781831  0.623490 0.937752 0.347305
2026-01-07  0.974928 -0.222521 0.988468 0.151428


# **FINAL FEATURES COLUMNS**

In [38]:
CATEGORY_COLS = [col for col in df.columns if col not in [
    'id_user', 'tanggal', 'gaji_bulanan', 'segmen_demografi_enc',
    'persona_dasar_enc', 'dow_sin', 'dow_cos', 'dom_sin', 'dom_cos'
]]

PROFILE_COLS  = ['gaji_bulanan', 'segmen_demografi_enc', 'persona_dasar_enc']
TEMPORAL_COLS = ['dow_sin', 'dow_cos', 'dom_sin', 'dom_cos']
FEATURE_COLS  = CATEGORY_COLS + PROFILE_COLS + TEMPORAL_COLS
N_FEATURES    = len(FEATURE_COLS)

print(f"Category columns  : {len(CATEGORY_COLS)} cols")
for c in CATEGORY_COLS:
    print(f"  - {c}")
print(f"\nProfile columns   : {len(PROFILE_COLS)} cols")
for c in PROFILE_COLS:
    print(f"  - {c}")
print(f"\nTemporal cyclical : {len(TEMPORAL_COLS)} cols")
for c in TEMPORAL_COLS:
    print(f"  - {c}")
print(f"\nTotal features    : {N_FEATURES}")

Category columns  : 10 cols
  - Belanja Online & Fashion
  - F&B dan Nongkrong
  - Groceries & Kebutuhan Pokok
  - Hiburan & Langganan
  - Investasi & Finansial
  - Kesehatan & Perawatan Diri
  - NLP Classified Transfer
  - Produktivitas & Digital
  - Tagihan & Utilitas
  - Transportasi

Profile columns   : 3 cols
  - gaji_bulanan
  - segmen_demografi_enc
  - persona_dasar_enc

Temporal cyclical : 4 cols
  - dow_sin
  - dow_cos
  - dom_sin
  - dom_cos

Total features    : 17


In [39]:
feat_path = OUTPUT_DIR / "feature_cols.json"
with open(feat_path, 'w') as f:
    json.dump(FEATURE_COLS, f, indent=2)
print(f"Saved: {feat_path.resolve()}")

Saved: /Users/faqihfirmanpratama/Documents/DBS Codingcamp/capstone/AI-ENGINEER/data/processed/02_lstm_ae_personalized_model/feature_cols.json


# **FILE INSPECTION & VERIFICATION**

In [43]:
print("Rata-rata pengeluaran harian per kategori:")

category_means = df[CATEGORY_COLS].mean().sort_values(ascending=False)
for cat, val in category_means.items():
    print(f"{cat:<30} : Rp {val:>7,.0f}")

Rata-rata pengeluaran harian per kategori:
Tagihan & Utilitas             : Rp  64,472
Kesehatan & Perawatan Diri     : Rp  35,056
Transportasi                   : Rp  30,293
Investasi & Finansial          : Rp  27,502
Hiburan & Langganan            : Rp  18,448
Belanja Online & Fashion       : Rp  17,963
Groceries & Kebutuhan Pokok    : Rp  15,901
F&B dan Nongkrong              : Rp   7,217
NLP Classified Transfer        : Rp   4,325
Produktivitas & Digital        : Rp   4,297


In [44]:
n_users_total = int(df['id_user'].nunique())


daily_path = OUTPUT_DIR / "daily_spending_matrix.csv"
df.to_csv(daily_path, index=False)
print(f"Saved: {daily_path.resolve()}")
print(f"kolom: {df.columns.tolist()}")
print(f"shape: {df.shape}")

Saved: /Users/faqihfirmanpratama/Documents/DBS Codingcamp/capstone/AI-ENGINEER/data/processed/02_lstm_ae_personalized_model/daily_spending_matrix.csv
kolom: ['tanggal', 'Belanja Online & Fashion', 'F&B dan Nongkrong', 'Groceries & Kebutuhan Pokok', 'Hiburan & Langganan', 'Investasi & Finansial', 'Kesehatan & Perawatan Diri', 'NLP Classified Transfer', 'Produktivitas & Digital', 'Tagihan & Utilitas', 'Transportasi', 'id_user', 'gaji_bulanan', 'segmen_demografi_enc', 'persona_dasar_enc', 'dow_sin', 'dow_cos', 'dom_sin', 'dom_cos']
shape: (40523, 19)


In [45]:
meta = {
    "window_size"          : WINDOW_SIZE,
    "n_features"           : N_FEATURES,
    "n_users"              : n_users_total,
    "feature_cols"         : FEATURE_COLS,
    "category_cols_count"  : len(CATEGORY_COLS),
    "profile_cols"         : PROFILE_COLS,
    "temporal_cols"        : TEMPORAL_COLS,
    "scaling_applied"      : False,
    "notes"                : "Raw values — no scaling applied. Scaler akan di-fit di notebook modeling."
}

meta_path = OUTPUT_DIR / "preprocessing_meta.json"
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print()
print("Output files:")
for fname in ['daily_spending_matrix.csv', 'feature_cols.json', 'preprocessing_meta.json']:
    fpath = OUTPUT_DIR / fname
    status = 'v' if fpath.exists() else 'X'
    print(f"  [{status}] {fpath.resolve()}")

print()
print(f"N_FEATURES  : {N_FEATURES}")
print(f"N_USERS     : {n_users_total}")
print(f"WINDOW_SIZE : {WINDOW_SIZE}")
print("\nPreprocessing selesai.")


Output files:
  [v] /Users/faqihfirmanpratama/Documents/DBS Codingcamp/capstone/AI-ENGINEER/data/processed/02_lstm_ae_personalized_model/daily_spending_matrix.csv
  [v] /Users/faqihfirmanpratama/Documents/DBS Codingcamp/capstone/AI-ENGINEER/data/processed/02_lstm_ae_personalized_model/feature_cols.json
  [v] /Users/faqihfirmanpratama/Documents/DBS Codingcamp/capstone/AI-ENGINEER/data/processed/02_lstm_ae_personalized_model/preprocessing_meta.json

N_FEATURES  : 17
N_USERS     : 500
WINDOW_SIZE : 14

Preprocessing selesai.
